<a href="https://colab.research.google.com/github/tanisha0710-ui/PMUY-AI/blob/main/Final_Cleaning_and_Naive_DiD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install dependencies and mount Drive
!pip install pyreadstat pandas numpy -q

import google.colab.drive as drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 18.4 MB/s eta 0:00:00
Mounted at /content/drive


In [3]:
import pandas as pd

file_path = "/content/drive/MyDrive/pmuy_data.csv"

df = pd.read_csv(file_path, low_memory=False)

df.shape

(1238208, 19)

In [4]:
!pip install pyarrow

In [5]:
import pandas as pd

# Original file
file_path = "/content/drive/MyDrive/pmuy_data.csv"

# Read CSV
df = pd.read_csv(file_path, low_memory=False)

# Check shape
print(df.shape)

# ---------------------------------------------------
# OPTION 1: Compress CSV using gzip (no data loss)
# ---------------------------------------------------

compressed_csv_path = "/content/drive/MyDrive/pmuy_data_compressed.csv.gz"

df.to_csv(
    compressed_csv_path,
    index=False,
    compression="gzip"
)

print("Compressed CSV saved successfully!")

(1238208, 19)
Compressed CSV saved successfully!


In [6]:
import pandas as pd

file_path = "/content/drive/MyDrive/pmuy_data_compressed.csv.gz"
df = pd.read_csv(file_path)

df.head()

,hhid,hv005,hv009,hv024,hv025,hv201,hv204,hv206,hv213,hv219,hv220,hv226,hv270,sh34,sh36,hv106_01,state,survey,post
0,1000101,191072,4,andaman and nicobar islands,urban,piped into dwelling,on premises,yes,cement,male,51,"lpg, natural gas",middle,christian,none of above,"no education, preschool",andaman and nicobar islands,4,0
1,1000109,191072,3,andaman and nicobar islands,urban,piped to yard/plot,on premises,yes,cement,female,40,"lpg, natural gas",richer,hindu,NaN,primary,andaman and nicobar islands,4,0
2,1000110,191072,4,andaman and nicobar islands,urban,piped to yard/plot,on premises,yes,cement,male,38,"lpg, natural gas",richer,hindu,none of above,secondary,andaman and nicobar islands,4,0
3,1000111,191072,3,andaman and nicobar islands,urban,piped into dwelling,on premises,yes,cement,male,46,"lpg, natural gas",richer,hindu,none of above,secondary,andaman and nicobar islands,4,0
4,1000117,191072,2,andaman and nicobar islands,urban,piped into dwelling,on premises,yes,cement,male,28,kerosene,middle,hindu,NaN,primary,andaman and nicobar islands,4,0


In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/pmuy_data_compressed.csv.gz')

print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nDtypes:")
print(df.dtypes)

Shape: (1238208, 19)

Column names:
['hhid', 'hv005', 'hv009', 'hv024', 'hv025', 'hv201', 'hv204', 'hv206', 'hv213', 'hv219', 'hv220', 'hv226', 'hv270', 'sh34', 'sh36', 'hv106_01', 'state', 'survey', 'post']

Dtypes:
hhid         int64
hv005        int64
hv009        int64
hv024       object
hv025       object
hv201       object
hv204       object
hv206       object
hv213       object
hv219       object
hv220       object
hv226       object
hv270       object
sh34        object
sh36        object
hv106_01    object
state       object
survey       int64
post         int64
dtype: object


In [8]:
print("Shape:", df.shape)
print("\nColumn names:")
print("First few rows:", df.head())

Shape: (1238208, 19)

Column names:
First few rows:       hhid   hv005  hv009                        hv024  hv025  \
0  1000101  191072      4  andaman and nicobar islands  urban   
1  1000109  191072      3  andaman and nicobar islands  urban   
2  1000110  191072      4  andaman and nicobar islands  urban   
3  1000111  191072      3  andaman and nicobar islands  urban   
4  1000117  191072      2  andaman and nicobar islands  urban   

                 hv201        hv204 hv206   hv213   hv219 hv220  \
0  piped into dwelling  on premises   yes  cement    male    51   
1   piped to yard/plot  on premises   yes  cement  female    40   
2   piped to yard/plot  on premises   yes  cement    male    38   
3  piped into dwelling  on premises   yes  cement    male    46   
4  piped into dwelling  on premises   yes  cement    male    28   

              hv226   hv270       sh34           sh36  \
0  lpg, natural gas  middle  christian  none of above   
1  lpg, natural gas  richer      hindu  

In [9]:
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_pct':   df.isna().mean() * 100
}).sort_values('missing_pct', ascending=False)

print(missing[missing['missing_count'] > 0])

      missing_count  missing_pct
sh36          51867     4.188876


In [10]:
print("=== sh36 (caste) value counts ===")
print(df['sh36'].value_counts(dropna=False))

print("\n=== sh34 (religion) value counts ===")
print(df['sh34'].value_counts(dropna=False))

print("\n=== Missing caste by round ===")
print(df.groupby('survey')['sh36'].apply(lambda x: x.isna().sum()))

print("\n=== Missing caste by state ===")
print(df.groupby('state')['sh36'].apply(lambda x: x.isna().mean()*100).sort_values(ascending=False).head(10))

=== sh36 (caste) value counts ===
sh36
other backward class    459929
none of above           249636
scheduled tribe         237543
scheduled caste         231436
NaN                      51867
don't know                7797
Name: count, dtype: int64

=== sh34 (religion) value counts ===
sh34
hindu                    930484
muslim                   145651
christian                 98840
sikh                      26939
buddhist/neo-buddhist     17602
other                     15686
jain                       1911
no religion                 897
parsi/zoroastrian           173
jewish                       25
Name: count, dtype: int64

=== Missing caste by round ===
survey
4    24227
5    27640
Name: sh36, dtype: int64

=== Missing caste by state ===
state
jammu and kashmir              29.192021
goa                            27.729384
assam                          23.550612
west bengal                    20.194546
tripura                        14.822084
andaman and nicobar islands    

In [11]:
# Using inplace parameter
df['sh36'].fillna('99', inplace=True)

print("Caste distribution after filling:")
print(df['sh36'].value_counts())
print(f"\nMissing remaining: {df['sh36'].isna().sum()}")

/tmp/ipykernel_4319/2469074944.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['sh36'].fillna('99', inplace=True)


Caste distribution after filling:
sh36
other backward class    459929
none of above           249636
scheduled tribe         237543
scheduled caste         231436
99                       51867
don't know                7797
Name: count, dtype: int64

Missing remaining: 0


In [12]:
df.isna().sum()

,0
hhid,0
hv005,0
hv009,0
hv024,0
hv025,0
hv201,0
hv204,0
hv206,0
hv213,0
hv219,0


In [22]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv('/content/drive/MyDrive/pmuy_data_compressed.csv.gz')

# ============================================================
# WEIGHTS: hv005 (DHS standard: divide by 1,000,000)
# ============================================================
df['weight'] = df['hv005'] / 1_000_000

print("=== WEIGHTS ===")
print(f"Weight range: {df['weight'].min():.4f} to {df['weight'].max():.4f}")
print(f"Weight mean: {df['weight'].mean():.4f}")
print(f"Sum of weights: {df['weight'].sum():.0f} (estimated population in household units)")
print(f"Sample size (N): {len(df):,}")
print(f"Design effect: {df['weight'].sum() / len(df):.2f}")  # How much weights inflate sample

# ============================================================
# CLEAN FUEL: hv226 → binary (CORRECT ORDER)
# ============================================================

CLEAN_FUELS = ['electricity', 'lpg, natural gas', 'biogas']

# Step 1: Drop households that don't cook food
df = df[~df['hv226'].isin(['no food cooked in house'])]

# Step 2: Create clean fuel binary (only for cooking households)
df['clean_fuel'] = df['hv226'].isin(CLEAN_FUELS).astype(int)

# Step 3: Drop any remaining NaN (if any exist)
df = df[df['clean_fuel'].notna()]

print("\n=== CLEAN FUEL ===")
print(df['clean_fuel'].value_counts())
print(f"\nClean fuel rate (unweighted): {df['clean_fuel'].mean()*100:.2f}%")

# Weighted clean fuel rate
weighted_clean = np.average(df['clean_fuel'], weights=df['weight']) * 100
print(f"Clean fuel rate (weighted): {weighted_clean:.2f}%")

=== WEIGHTS ===
Weight range: 0.0007 to 38.9548
Weight mean: 1.0000
Sum of weights: 1238208 (estimated population in household units)
Sample size (N): 1,238,208
Design effect: 1.00

=== CLEAN FUEL ===
clean_fuel
0    671767
1    564185
Name: count, dtype: int64

Clean fuel rate (unweighted): 45.65%
Clean fuel rate (weighted): 51.53%


In [23]:
# ============================================================
# CHECK STATES ACROSS ROUNDS
# ============================================================
print("\n" + "="*60)
print("STATE CONSISTENCY CHECK")
print("="*60)

states_nfhs4 = set(df[df['post']==0]['state'].unique())
states_nfhs5 = set(df[df['post']==1]['state'].unique())

print(f"States in NFHS-4: {len(states_nfhs4)}")
print(f"States in NFHS-5: {len(states_nfhs5)}")

states_in_both = states_nfhs4.intersection(states_nfhs5)
print(f"\nStates in both: {len(states_in_both)}")

states_only_4 = states_nfhs4 - states_nfhs5
if states_only_4:
    print(f"\n⚠️ States only in NFHS-4: {sorted(states_only_4)}")

states_only_5 = states_nfhs5 - states_nfhs4
if states_only_5:
    print(f"\n⚠️ States only in NFHS-5: {sorted(states_only_5)}")

if not states_only_4 and not states_only_5:
    print("\n✓ Perfect! All states in both rounds")


STATE CONSISTENCY CHECK
States in NFHS-4: 35
States in NFHS-5: 35

States in both: 35

✓ Perfect! All states in both rounds


In [24]:
import pandas as pd
import numpy as np

print("="*60)
print("CREATING BINARY CONTROL VARIABLES")
print("="*60)

# ============================================================
# 1. RURAL (binary) - from hv025
# ============================================================
if 'hv025' in df.columns:
    df['rural'] = df['hv025'].str.lower().str.strip().map({
        'rural': 1,
        'urban': 0
    }).fillna(0).astype(int)
    print("✓ Created 'rural' (1=rural, 0=urban)")

# ============================================================
# 2. ELECTRICITY (binary) - from hv206
# ============================================================
if 'hv206' in df.columns:
    df['electricity'] = df['hv206'].str.lower().str.strip().map({
        'yes': 1,
        'no': 0
    }).fillna(0).astype(int)
    print("✓ Created 'electricity' (1=has electricity, 0=no)")

# ============================================================
# 3. FEMALE HEAD (binary) - from hv219
# ============================================================
if 'hv219' in df.columns:
    df['female_head'] = df['hv219'].str.lower().str.strip().map({
        'female': 1,
        'male': 0
    }).fillna(0).astype(int)
    print("✓ Created 'female_head' (1=female head, 0=male head)")

# ============================================================
# 4. IMPROVED WATER (binary) - from hv201
# ============================================================
if 'hv201' in df.columns:
    improved_water_sources = [
        'piped into dwelling', 'piped to yard/plot', 'public tap/standpipe',
        'tube well or borehole', 'protected well', 'protected spring',
        'rainwater', 'bottled water'
    ]
    df['improved_water'] = df['hv201'].str.lower().str.strip().isin(improved_water_sources).astype(int)
    print("✓ Created 'improved_water' (1=improved source, 0=unimproved)")

# ============================================================
# 5. IMPROVED FLOOR (binary) - from hv213
# ============================================================
if 'hv213' in df.columns:
    unimproved_floors = [
        'mud/clay/earth', 'dung', 'sand', 'raw wood planks',
        'palm, bamboo', 'stone'
    ]
    df['improved_floor'] = (~df['hv213'].str.lower().str.strip().isin(unimproved_floors)).astype(int)
    print("✓ Created 'improved_floor' (1=improved, 0=unimproved)")

# ============================================================
# 6. CASTE DUMBIRS - from sh36
# ============================================================
if 'sh36' in df.columns:
    # Fill missing as 'not stated'
    df['caste'] = df['sh36'].fillna('not stated').str.lower().str.strip()

    # Create dummies
    caste_dummies = pd.get_dummies(df['caste'], prefix='caste', drop_first=False)

    # Select relevant caste categories
    caste_categories = ['scheduled caste', 'scheduled tribe', 'other backward class', 'none of above']
    for category in caste_categories:
        col_name = f'caste_{category.replace(" ", "_")}'
        if col_name in caste_dummies.columns:
            df[col_name] = caste_dummies[col_name].astype(int)
        else:
            df[f'caste_{category.replace(" ", "_")}'] = 0

    print("✓ Created caste dummies: 'caste_scheduled_caste', 'caste_scheduled_tribe', 'caste_other_backward_class', 'caste_none_of_above'")

# ============================================================
# 7. RELIGION DUMBIRS - from sh34
# ============================================================
if 'sh34' in df.columns:
    df['religion'] = df['sh34'].str.lower().str.strip()

    # Create dummies for major religions
    religion_dummies = pd.get_dummies(df['religion'], prefix='religion')

    major_religions = ['hindu', 'muslim', 'christian', 'sikh', 'buddhist']
    for religion in major_religions:
        col_name = f'religion_{religion}'
        if col_name in religion_dummies.columns:
            df[col_name] = religion_dummies[col_name].astype(int)
        else:
            df[col_name] = 0

    print("✓ Created religion dummies: 'religion_hindu', 'religion_muslim', 'religion_christian', 'religion_sikh', 'religion_buddhist'")

# ============================================================
# 8. EDUCATION OF HEAD (binary for higher education) - from hv106_01
# ============================================================
if 'hv106_01' in df.columns:
    df['head_edu'] = df['hv106_01'].str.lower().str.strip()
    # Higher education = secondary or above
    df['head_higher_edu'] = df['head_edu'].isin(['secondary', 'higher']).astype(int)
    # At least primary education
    df['head_primary_edu'] = df['head_edu'].isin(['primary', 'secondary', 'higher']).astype(int)
    print("✓ Created 'head_higher_edu' (1=secondary+), 'head_primary_edu' (1=primary+)")

# ============================================================
# 9. PIPED WATER (specific binary) - from hv201
# ============================================================
if 'hv201' in df.columns:
    piped_sources = ['piped into dwelling', 'piped to yard/plot']
    df['piped_water'] = df['hv201'].str.lower().str.strip().isin(piped_sources).astype(int)
    print("✓ Created 'piped_water' (1=piped to dwelling/yard, 0=other)")

# ============================================================
# 10. WEALTH QUINTILE (keep as numeric, but create rich dummy)
# ============================================================
if 'hv270' in df.columns:
    # Convert wealth to numeric if needed
    wealth_map = {
        'poorest': 1, 'poorer': 2, 'middle': 3, 'richer': 4, 'richest': 5
    }
    if df['hv270'].dtype == 'object':
        df['wealth_quintile'] = df['hv270'].str.lower().str.strip().map(wealth_map)
    else:
        df['wealth_quintile'] = pd.to_numeric(df['hv270'], errors='coerce')

    # Top 2 quintiles as rich
    df['rich'] = (df['wealth_quintile'] >= 4).astype(int)
    print("✓ Created 'wealth_quintile' (1-5) and 'rich' (1=quintile 4-5)")

# ============================================================
# VERIFY CREATED VARIABLES
# ============================================================
print("\n" + "="*60)
print("CREATED BINARY VARIABLES SUMMARY")
print("="*60)

binary_vars = ['rural', 'electricity', 'female_head', 'improved_water',
               'improved_floor', 'head_higher_edu', 'head_primary_edu',
               'piped_water', 'rich']

for var in binary_vars:
    if var in df.columns:
        print(f"\n{var}:")
        print(f"  Mean: {df[var].mean()*100:.1f}%")
        print(f"  Value counts:\n{df[var].value_counts().to_dict()}")

# ============================================================
# LIST ALL CONTROLS FOR REGRESSION
# ============================================================
print("\n" + "="*60)
print("CONTROL VARIABLES FOR DiD REGRESSION")
print("="*60)

control_vars = []
for var in ['rural', 'electricity', 'female_head', 'improved_water',
            'improved_floor', 'head_higher_edu', 'piped_water', 'rich',
            'wealth_quintile', 'caste_scheduled_caste', 'caste_scheduled_tribe',
            'caste_other_backward_class', 'religion_muslim', 'religion_christian']:
    if var in df.columns:
        control_vars.append(var)

print(f"Available controls: {len(control_vars)}")
for i, var in enumerate(control_vars, 1):
    print(f"  {i}. {var}")

# ============================================================
# OPTIONAL: SAVE DATAFRAME WITH NEW VARIABLES
# ============================================================
df.to_csv('data_with_binary_controls.csv', index=False)
print("\n✓ Saved: data_with_binary_controls.csv")

CREATING BINARY CONTROL VARIABLES
✓ Created 'rural' (1=rural, 0=urban)
✓ Created 'electricity' (1=has electricity, 0=no)
✓ Created 'female_head' (1=female head, 0=male head)
✓ Created 'improved_water' (1=improved source, 0=unimproved)
✓ Created 'improved_floor' (1=improved, 0=unimproved)
✓ Created caste dummies: 'caste_scheduled_caste', 'caste_scheduled_tribe', 'caste_other_backward_class', 'caste_none_of_above'
✓ Created religion dummies: 'religion_hindu', 'religion_muslim', 'religion_christian', 'religion_sikh', 'religion_buddhist'
✓ Created 'head_higher_edu' (1=secondary+), 'head_primary_edu' (1=primary+)
✓ Created 'piped_water' (1=piped to dwelling/yard, 0=other)
✓ Created 'wealth_quintile' (1-5) and 'rich' (1=quintile 4-5)

CREATED BINARY VARIABLES SUMMARY

rural:
  Mean: 72.9%
  Value counts:
{1: 900949, 0: 335003}

electricity:
  Mean: 92.4%
  Value counts:
{1: 1141833, 0: 94119}

female_head:
  Mean: 15.9%
  Value counts:
{0: 1039530, 1: 196422}

improved_water:
  Mean: 83.0%
 

In [25]:
 # ============================================================
# CHECK UNIQUE STATES (Debug the 36 vs 35 issue)
# ============================================================

print("="*60)
print("DEBUG: CHECKING UNIQUE STATES")
print("="*60)

unique_states = df['state'].unique()
print(f"Total unique states in dataset: {len(unique_states)}")
print(f"States: {sorted(unique_states)}")

# Check if any state has both '37' and 'odisha' or similar issues
for state in unique_states:
    print(f"  '{state}' - Type: {type(state)}")

DEBUG: CHECKING UNIQUE STATES
Total unique states in dataset: 35
States: ['andaman and nicobar islands', 'andhra pradesh', 'arunachal pradesh', 'assam', 'bihar', 'chandigarh', 'chhattisgarh', 'dadra and nagar haveli and daman and diu', 'delhi', 'goa', 'gujarat', 'haryana', 'himachal pradesh', 'jammu and kashmir', 'jharkhand', 'karnataka', 'kerala', 'lakshadweep', 'madhya pradesh', 'maharashtra', 'manipur', 'meghalaya', 'mizoram', 'nagaland', 'odisha', 'puducherry', 'punjab', 'rajasthan', 'sikkim', 'tamil nadu', 'telangana', 'tripura', 'uttar pradesh', 'uttarakhand', 'west bengal']
  'andaman and nicobar islands' - Type: <class 'str'>
  'andhra pradesh' - Type: <class 'str'>
  'arunachal pradesh' - Type: <class 'str'>
  'assam' - Type: <class 'str'>
  'bihar' - Type: <class 'str'>
  'chandigarh' - Type: <class 'str'>
  'chhattisgarh' - Type: <class 'str'>
  'dadra and nagar haveli and daman and diu' - Type: <class 'str'>
  'goa' - Type: <class 'str'>
  'gujarat' - Type: <class 'str'>
  

In [26]:
# DEBUG: Find which state has different names across rounds
print("="*60)
print("DEBUG: FINDING STATE NAME MISMATCH")
print("="*60)

# Get states in NFHS-4 (post=0)
states_nfhs4 = set(df[df['post'] == 0]['state'].unique())
print(f"States in NFHS-4: {len(states_nfhs4)}")
print(f"NFHS-4 states: {sorted(states_nfhs4)}")

# Get states in NFHS-5 (post=1)
states_nfhs5 = set(df[df['post'] == 1]['state'].unique())
print(f"\nStates in NFHS-5: {len(states_nfhs5)}")
print(f"NFHS-5 states: {sorted(states_nfhs5)}")

# Find mismatches
only_in_nfhs4 = states_nfhs4 - states_nfhs5
only_in_nfhs5 = states_nfhs5 - states_nfhs4

print(f"\nStates ONLY in NFHS-4: {only_in_nfhs4}")
print(f"States ONLY in NFHS-5: {only_in_nfhs5}")

# Show sample counts for mismatched states
print("\n" + "="*60)
print("DETAILS OF MISMATCHED STATES")
print("="*60)

for state in only_in_nfhs4:
    n_count = len(df[(df['state'] == state)])
    print(f"'{state}' appears in {n_count} rows total")
    print(f"  - NFHS-4: {len(df[(df['state'] == state) & (df['post'] == 0)])}")
    print(f"  - NFHS-5: {len(df[(df['state'] == state) & (df['post'] == 1)])}")

for state in only_in_nfhs5:
    n_count = len(df[(df['state'] == state)])
    print(f"'{state}' appears in {n_count} rows total")
    print(f"  - NFHS-4: {len(df[(df['state'] == state) & (df['post'] == 0)])}")
    print(f"  - NFHS-5: {len(df[(df['state'] == state) & (df['post'] == 1)])}")

DEBUG: FINDING STATE NAME MISMATCH
States in NFHS-4: 35
NFHS-4 states: ['andaman and nicobar islands', 'andhra pradesh', 'arunachal pradesh', 'assam', 'bihar', 'chandigarh', 'chhattisgarh', 'dadra and nagar haveli and daman and diu', 'delhi', 'goa', 'gujarat', 'haryana', 'himachal pradesh', 'jammu and kashmir', 'jharkhand', 'karnataka', 'kerala', 'lakshadweep', 'madhya pradesh', 'maharashtra', 'manipur', 'meghalaya', 'mizoram', 'nagaland', 'odisha', 'puducherry', 'punjab', 'rajasthan', 'sikkim', 'tamil nadu', 'telangana', 'tripura', 'uttar pradesh', 'uttarakhand', 'west bengal']

States in NFHS-5: 35
NFHS-5 states: ['andaman and nicobar islands', 'andhra pradesh', 'arunachal pradesh', 'assam', 'bihar', 'chandigarh', 'chhattisgarh', 'dadra and nagar haveli and daman and diu', 'delhi', 'goa', 'gujarat', 'haryana', 'himachal pradesh', 'jammu and kashmir', 'jharkhand', 'karnataka', 'kerala', 'lakshadweep', 'madhya pradesh', 'maharashtra', 'manipur', 'meghalaya', 'mizoram', 'nagaland', 'odi

In [27]:
# ============================================================
# FIX WEALTH QUINTILE (Convert hv270 to numeric)
# ============================================================

print("="*60)
print("FIXING WEALTH QUINTILE")
print("="*60)

# Map wealth categories to numeric values
wealth_map = {
    'poorest': 1,
    'poorer': 2,
    'middle': 3,
    'richer': 4,
    'richest': 5
}

# Convert hv270 to numeric wealth quintile
df['wealth_quintile'] = df['hv270'].astype(str).str.lower().map(wealth_map)

print(f"Wealth quintile range: {df['wealth_quintile'].min():.0f} to {df['wealth_quintile'].max():.0f}")
print(f"Wealth quintile mean: {df['wealth_quintile'].mean():.2f}")

# Create rich dummy (top 2 quintiles)
df['rich'] = (df['wealth_quintile'] >= 4).astype(int)

print(f"Rich (Q4-Q5) rate: {df['rich'].mean()*100:.1f}%")

FIXING WEALTH QUINTILE
Wealth quintile range: 1 to 5
Wealth quintile mean: 2.85
Rich (Q4-Q5) rate: 35.2%


In [28]:
# ============================================================
# PART 1: DEFINE TREATMENT (Based on NFHS-4 median)
# ============================================================

print("\n" + "="*60)
print("STEP 1: DEFINING TREATMENT (High Exposure)")
print("="*60)

# Calculate state-level weighted clean fuel for NFHS-4 only
state_nfhs4 = df[df['post'] == 0].groupby('state').apply(
    lambda x: np.average(x['clean_fuel'], weights=x['weight']) * 100,
    include_groups=False
).sort_values()

print(f"Number of states in NFHS-4: {len(state_nfhs4)}")

# Find median
median_value = state_nfhs4.median()

# Define treatment (below median = high exposure)
treatment_states = state_nfhs4[state_nfhs4 < median_value].index.tolist()

# Add treatment indicator
df['high_exposure'] = df['state'].isin(treatment_states).astype(int)

print(f"Median NFHS-4 clean fuel: {median_value:.1f}%")
print(f"Treatment states (below median): {len(treatment_states)}")
print(f"Control states (above median): {len(state_nfhs4) - len(treatment_states)}")


STEP 1: DEFINING TREATMENT (High Exposure)
Number of states in NFHS-4: 35
Median NFHS-4 clean fuel: 52.3%
Treatment states (below median): 17
Control states (above median): 18


In [29]:
# Which states ended up in which group?
state_check = (
    df[df['post'] == 0]
    .groupby('state')
    .apply(lambda x: np.average(x['clean_fuel'], weights=x['weight']) * 100,
           include_groups=False)
    .reset_index()
    .rename(columns={0: 'baseline_clean_pct'})
    .sort_values('baseline_clean_pct')
)

median_val = state_check['baseline_clean_pct'].median()
state_check['high_exposure'] = (state_check['baseline_clean_pct'] < median_val).astype(int)

print("=== ALL STATES SORTED BY BASELINE CLEAN FUEL ===")
print(state_check.to_string(index=False))
print(f"\nMedian: {median_val:.1f}%")
print(f"\nTreatment (high_exposure=1, PMUY targets):")
print(state_check[state_check['high_exposure']==1]['state'].tolist())
print(f"\nControl (high_exposure=0):")
print(state_check[state_check['high_exposure']==0]['state'].tolist())

=== ALL STATES SORTED BY BASELINE CLEAN FUEL ===
                                   state  baseline_clean_pct  high_exposure
                                   bihar           17.760521              1
                               jharkhand           18.959144              1
                                  odisha           19.168095              1
                               meghalaya           21.842860              1
                            chhattisgarh           22.780439              1
                                   assam           25.085946              1
                             west bengal           27.867793              1
                          madhya pradesh           29.595975              1
                                 tripura           31.867841              1
                               rajasthan           31.872670              1
                             lakshadweep           31.909577              1
                           uttar prades

In [30]:
# ============================================================
# PART 2: SUMMARY STATISTICS (RAW MEANS)
# ============================================================

print("\n" + "="*80)
print("STEP 2: COMPREHENSIVE SUMMARY STATISTICS (RAW MEANS)")
print("="*80)

# Define groups - VERIFY THESE ARE CORRECT
# post = 0 is NFHS-4 (Pre), post = 1 is NFHS-5 (Post)
# high_exposure = 1 is Treatment (low baseline), high_exposure = 0 is Control (high baseline)
groups = [
    ('Pre-Treatment', 0, 1),   # Pre (NFHS-4), Treatment (high_exposure=1)
    ('Pre-Control', 0, 0),     # Pre (NFHS-4), Control (high_exposure=0)
    ('Post-Treatment', 1, 1),  # Post (NFHS-5), Treatment (high_exposure=1)
    ('Post-Control', 1, 0)     # Post (NFHS-5), Control (high_exposure=0)
]

# Define variables to summarize (NO PERCENTAGE CONVERSION)
variables = {
    'Clean Fuel (0/1)': 'clean_fuel',      # Raw mean = proportion (0-1)
    'Household Size': 'hv009',
    'Head Age': 'hv220',
    'Wealth Quintile (1-5)': 'wealth_quintile',
    'Rural (0/1)': 'rural',               # Raw mean = proportion rural
    'Electricity (0/1)': 'electricity',   # Raw mean = proportion with electricity
    'Female Head (0/1)': 'female_head',   # Raw mean = proportion female-headed
    'Improved Water (0/1)': 'improved_water',
    'Improved Floor (0/1)': 'improved_floor',
    'Piped Water (0/1)': 'piped_water',
    'Rich Q4-Q5 (0/1)': 'rich',
    'Head Higher Edu (0/1)': 'head_higher_edu',
}

# Function to safely convert to numeric
def to_numeric_safe(series):
    return pd.to_numeric(series, errors='coerce')

# Prepare dataframe
df_clean = df.copy()
for var_col in variables.values():
    if var_col in df_clean.columns and var_col is not None:
        df_clean[var_col] = to_numeric_safe(df_clean[var_col])

# Store results
results = {'mean': {}, 'std': {}, 'min': {}, 'max': {}}

for var_name, var_col in variables.items():
    if var_col is None or var_col not in df_clean.columns:
        print(f"Warning: '{var_name}' column '{var_col}' not found - skipping")
        continue

    for stat_name in ['mean', 'std', 'min', 'max']:
        results[stat_name][var_name] = {}

    for label, post_val, treat_val in groups:
        subset = df_clean[(df_clean['post'] == post_val) & (df_clean['high_exposure'] == treat_val)]

        values = df_clean.loc[subset.index, var_col]
        valid_values = values.dropna()
        valid_weights = subset.loc[valid_values.index, 'weight'] if len(valid_values) > 0 else []

        if len(valid_values) > 0:
            # Weighted mean (RAW - no percentage conversion)
            weighted_mean = np.average(valid_values, weights=valid_weights)
            results['mean'][var_name][label] = weighted_mean

            # Weighted standard deviation
            variance = np.average((valid_values - weighted_mean)**2, weights=valid_weights)
            weighted_std = np.sqrt(variance)
            results['std'][var_name][label] = weighted_std

            # Min and Max
            results['min'][var_name][label] = valid_values.min()
            results['max'][var_name][label] = valid_values.max()
        else:
            for stat_name in ['mean', 'std', 'min', 'max']:
                results[stat_name][var_name][label] = np.nan

# Print combined summary table (RAW MEANS)
print("\nCOMBINED SUMMARY (Mean ± Std Dev) - RAW VALUES")
print("="*100)

combined_data = {'Variable': []}
for label, _, _ in groups:
    combined_data[label] = []

for var_name in variables.keys():
    if var_name not in results['mean']:
        continue
    combined_data['Variable'].append(var_name)
    for label, _, _ in groups:
        mean_val = results['mean'][var_name].get(label, np.nan)
        std_val = results['std'][var_name].get(label, np.nan)
        if pd.isna(mean_val) or pd.isna(std_val):
            formatted = 'N/A'
        else:
            # For binary variables (0/1), show as proportion (e.g., 0.284)
            # For continuous variables, show as is
            if '0/1' in var_name or var_name in ['Rural (0/1)', 'Electricity (0/1)', 'Female Head (0/1)',
                                                   'Improved Water (0/1)', 'Improved Floor (0/1)',
                                                   'Piped Water (0/1)', 'Rich Q4-Q5 (0/1)',
                                                   'Head Higher Edu (0/1)', 'Clean Fuel (0/1)']:
                formatted = f"{mean_val:.3f} ± {std_val:.3f}"
            else:
                formatted = f"{mean_val:.2f} ± {std_val:.2f}"
        combined_data[label].append(formatted)

# Add observations
combined_data['Variable'].append('Observations (N)')
for label, post_val, treat_val in groups:
    subset = df[(df['post'] == post_val) & (df['high_exposure'] == treat_val)]
    combined_data[label].append(f"{len(subset):,}")

combined_table = pd.DataFrame(combined_data)
print(combined_table.to_string(index=False))

# Print key finding (Clean Fuel Rate in PERCENTAGE for interpretation)
print("\n" + "="*80)
print("KEY FINDING - Clean Fuel Rates (Converted to Percentage for Interpretation)")
print("="*80)

pre_treatment = results['mean']['Clean Fuel (0/1)']['Pre-Treatment'] * 100
post_treatment = results['mean']['Clean Fuel (0/1)']['Post-Treatment'] * 100
pre_control = results['mean']['Clean Fuel (0/1)']['Pre-Control'] * 100
post_control = results['mean']['Clean Fuel (0/1)']['Post-Control'] * 100

print(f"Treatment Group (High Exposure): {pre_treatment:.1f}% → {post_treatment:.1f}% (Change: +{post_treatment - pre_treatment:.1f}pp)")
print(f"Control Group (Low Exposure):   {pre_control:.1f}% → {post_control:.1f}% (Change: {post_control - pre_control:.1f}pp)")
print(f"\nDiD Effect: {(post_treatment - pre_treatment) - (post_control - pre_control):.1f} percentage points")

# Also show raw proportions
print("\n" + "="*80)
print("VERIFICATION - Group Definitions")
print("="*80)
print("Pre = NFHS-4 (post=0)")
print("Post = NFHS-5 (post=1)")
print("Treatment = High Exposure states (below median NFHS-4 clean fuel)")
print("Control = Low Exposure states (above/equal median NFHS-4 clean fuel)")


STEP 2: COMPREHENSIVE SUMMARY STATISTICS (RAW MEANS)

COMBINED SUMMARY (Mean ± Std Dev) - RAW VALUES
             Variable Pre-Treatment   Pre-Control Post-Treatment  Post-Control
     Clean Fuel (0/1) 0.274 ± 0.446 0.630 ± 0.483  0.420 ± 0.494 0.795 ± 0.404
       Household Size   4.98 ± 2.43   4.37 ± 2.06    4.74 ± 2.26   4.11 ± 2.00
             Head Age 47.63 ± 14.19 49.33 ± 13.72  48.68 ± 14.18 51.13 ± 13.81
Wealth Quintile (1-5)   2.49 ± 1.38   3.60 ± 1.22    2.52 ± 1.38   3.54 ± 1.25
          Rural (0/1) 0.755 ± 0.430 0.532 ± 0.499  0.759 ± 0.428 0.557 ± 0.497
    Electricity (0/1) 0.805 ± 0.396 0.971 ± 0.167  0.949 ± 0.221 0.986 ± 0.117
    Female Head (0/1) 0.147 ± 0.354 0.145 ± 0.353  0.168 ± 0.374 0.182 ± 0.386
 Improved Water (0/1) 0.920 ± 0.271 0.931 ± 0.254  0.824 ± 0.381 0.733 ± 0.442
 Improved Floor (0/1) 0.439 ± 0.496 0.772 ± 0.420  0.519 ± 0.500 0.806 ± 0.396
    Piped Water (0/1) 0.154 ± 0.361 0.472 ± 0.499  0.195 ± 0.397 0.494 ± 0.500
     Rich Q4-Q5 (0/1) 0.261 ±